## Exercise 1: GLMs and Poisson Regression
**Dataset Used:** Bike Sharing

1. Train standard linear regression. Check residuals (QQ-Plot).
2. Train PoissonRegressor. Compare MAE and Mean Poisson Deviance.
3. Implement Negative Binomial Regression with statsmodels.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, PoissonRegressor
from sklearn.metrics import mean_absolute_error, mean_poisson_deviance
from sklearn.preprocessing import StandardScaler

# Download Bike Sharing Dataset directly
url = "https://raw.githubusercontent.com/christophM/interpretable-ml-book/master/data/bike.csv"
df = pd.read_csv(url)

# Simple preprocessing
features = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']
X = df[features]
y = df['cnt']

# Convert categorical to dummies
X = pd.get_dummies(X, columns=['season', 'weathersit'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Standard Linear Regression
lin_reg = LinearRegression().fit(X_train_scaled, y_train)
y_pred_lin = lin_reg.predict(X_test_scaled)

residuals = y_test - y_pred_lin
plt.figure(figsize=(6,4))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot: Linear Regression Residuals")
plt.show()
print("Observation: The residuals are not perfectly normally distributed, especially at the tails, which violates linear regression assumptions.")

# 2. Poisson Regressor
pois_reg = PoissonRegressor(alpha=1e-4, max_iter=300).fit(X_train_scaled, y_train)
y_pred_pois = pois_reg.predict(X_test_scaled)

# Prevent negative predictions for deviance metric in lin reg
y_pred_lin = np.clip(y_pred_lin, a_min=1, a_max=None)

mae_lin = mean_absolute_error(y_test, y_pred_lin)
mae_pois = mean_absolute_error(y_test, y_pred_pois)
dev_lin = mean_poisson_deviance(y_test, y_pred_lin)
dev_pois = mean_poisson_deviance(y_test, y_pred_pois)

print(f"Linear Regression  - MAE: {mae_lin:.2f}, Poisson Deviance: {dev_lin:.2f}")
print(f"Poisson Regression - MAE: {mae_pois:.2f}, Poisson Deviance: {dev_pois:.2f}")

# 3. Negative Binomial with statsmodels
# Add constant for statsmodels
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

nb_model = sm.GLM(y_train, X_train_sm, family=sm.families.NegativeBinomial(alpha=1.0)).fit()
y_pred_nb = nb_model.predict(X_test_sm)

mae_nb = mean_absolute_error(y_test, y_pred_nb)
print(f"Negative Binomial  - MAE: {mae_nb:.2f}")
print("Negative Binomial is better when data is overdispersed (variance > mean), which is common in count data.")
